# Bird Call Classifier — Kaggle GPU Training

This notebook trains the BirdVGG model on a Kaggle GPU using the existing project scripts.

**Before running:**
1. Upload your data as a Kaggle dataset (see README *Training on Kaggle GPU*).
2. Add that dataset to this notebook (*Add Data*).
3. Enable GPU: *Settings → Accelerator → GPU*.
4. Set `DATASET_SLUG` in the next cell to match your uploaded dataset folder name under `/kaggle/input/`.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/Dewaszmi/bird-call-classifier.git"
REPO_DIR = Path("/kaggle/working/bird-call-classifier")
INPUT_DIR = Path("/kaggle/input")

# Folder name under /kaggle/input/ after you add your dataset to the notebook.
DATASET_SLUG = "bird-call-processed"

BATCHING_STRATEGY = "none"  # none | no-batch | length-bucketing | masked-gap
EPOCHS = 150
BATCH_SIZE = 32
RUN_NAME = f"kaggle-{BATCHING_STRATEGY}"

In [ ]:
def find_processed_data() -> Path:
    candidates: list[Path] = []

    preferred = INPUT_DIR / DATASET_SLUG / "processed_data"
    if preferred.is_dir():
        candidates.append(preferred)

    for path in INPUT_DIR.glob("*/processed_data"):
        if path.is_dir():
            candidates.append(path)

    if not candidates:
        available = sorted(p.name for p in INPUT_DIR.iterdir())
        raise FileNotFoundError(
            "No processed_data found under /kaggle/input/.\n"
            f"Available input folders: {available}\n"
            "Add your packaged dataset to this notebook and set DATASET_SLUG."
        )

    return candidates[0]


def clone_repo() -> None:
    if REPO_DIR.exists():
        print(f"Repo already present at {REPO_DIR}")
        return

    subprocess.run(
        ["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)],
        check=True,
    )
    print(f"Cloned repo to {REPO_DIR}")


def link_processed_data(source: Path) -> Path:
    target = REPO_DIR / "processed_data"
    if target.exists() or target.is_symlink():
        target.unlink()
    target.symlink_to(source)
    print(f"Linked {target} -> {source}")
    return target


clone_repo()
processed_data = find_processed_data()
link_processed_data(processed_data)
os.chdir(REPO_DIR)
print(f"Working directory: {Path.cwd()}")

In [ ]:
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-kaggle.txt"],
    check=True,
)

import torch

if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU not available. Enable GPU in notebook settings.")

print(f"PyTorch {torch.__version__}")
print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
train_cmd = [
    sys.executable,
    "scripts/train.py",
    "--batching-strategy", BATCHING_STRATEGY,
    "--epochs", str(EPOCHS),
    "--batch-size", str(BATCH_SIZE),
    "--device", "cuda",
    "--run-name", RUN_NAME,
]

print(" ".join(train_cmd))
result = subprocess.run(train_cmd)
if result.returncode != 0:
    raise RuntimeError(f"Training failed with exit code {result.returncode}")

In [ ]:
import shutil

output_dir = Path("/kaggle/working/outputs") / BATCHING_STRATEGY
output_dir.mkdir(parents=True, exist_ok=True)

for name in ("best.pt", "history.json"):
    src = REPO_DIR / "checkpoints" / name
    if src.exists():
        shutil.copy2(src, output_dir / name)
        print(f"Saved {output_dir / name}")

runs_src = REPO_DIR / "runs" / RUN_NAME
if runs_src.exists():
    shutil.copytree(runs_src, output_dir / "runs", dirs_exist_ok=True)
    print(f"Saved TensorBoard logs to {output_dir / 'runs'}")

print("\nOutputs are in /kaggle/working/outputs and will be available after the notebook finishes.")

## Optional: preprocess raw audio on Kaggle

Only needed if you uploaded raw audio (`data/`) instead of `processed_data/`.
Uncomment the cell below, set `DATASET_SLUG` to your raw-audio dataset, then run it before training.

In [ ]:
# # Uncomment if you uploaded raw audio instead of processed spectrograms.
# raw_source = INPUT_DIR / DATASET_SLUG / "data"
# if not raw_source.is_dir():
#     raw_source = next(INPUT_DIR.glob("*/data"))
# raw_target = REPO_DIR / "data"
# if raw_target.exists() or raw_target.is_symlink():
#     raw_target.unlink()
# raw_target.symlink_to(raw_source)
# subprocess.run([sys.executable, "scripts/preprocess_data.py"], check=True)

## Optional: train all four batching strategies

This runs four full training jobs (150 epochs each). Expect several hours of GPU time.

In [ ]:
# Uncomment to run all four batching strategies sequentially.
# result = subprocess.run([sys.executable, "train_all_modes.py"])
# if result.returncode != 0:
#     raise RuntimeError(f"train_all_modes.py failed with exit code {result.returncode}")